# 04. Feeder Conductor Losses & Empirical Proof of Loss ∝ I²

## Electrical Engineering Context
In electrical distribution networks, technical losses consist primarily of:
1. **Conductor Copper / Joule Heating Losses:**
   $$P_{loss} = 3 \cdot I_{avg}^2 \cdot R_{feeder} = (I_r^2 + I_y^2 + I_b^2) \cdot R_{feeder}$$
2. **Transformer Core (Iron) and Copper Losses:**
   $$P_{loss\_TR} = P_0 + \left(\frac{S}{S_{rated}}\right)^2 P_k$$

In this notebook, we empirically investigate whether the generated telemetry strictly adheres to the physical quadratic law $\text{Loss} \propto I^2$, rank feeders by total energy dissipation, and analyze the financial impact of line resistance.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.utils.config import PROCESSED_DATA_DIR

df_meas = pd.read_csv(PROCESSED_DATA_DIR / "fact_electrical_measurements.csv")
dim_fdr = pd.read_csv(PROCESSED_DATA_DIR / "dim_feeder.csv")


## 1. Feeder Conductor Loss Ranking
We aggregate cumulative energy delivery vs technical dissipation.


In [ ]:
loss_summary = df_meas.groupby("feeder_id").agg({
    "energy_kwh": "sum",
    "feeder_loss_kw": lambda s: (s * 0.5).sum(),
    "feeder_loss_pct": "mean",
    "current_avg": "mean"
}).reset_index()

loss_summary["Delivered_MWh"] = (loss_summary["energy_kwh"] / 1000.0).round(2)
loss_summary["Loss_MWh"] = (loss_summary["feeder_loss_kw"] / 1000.0).round(2)
loss_summary["Avg_Loss_Pct"] = loss_summary["feeder_loss_pct"].round(2)

loss_summary = loss_summary.merge(dim_fdr[["feeder_id", "feeder_name", "length_km", "total_resistance_ohm"]], on="feeder_id")
loss_summary = loss_summary.sort_values(by="Avg_Loss_Pct", ascending=False)

print("Feeder Technical Loss Scorecard:")
print(loss_summary[["feeder_id", "feeder_name", "length_km", "total_resistance_ohm", "Delivered_MWh", "Loss_MWh", "Avg_Loss_Pct"]])


## 2. Empirical Verification of Physical Law: Loss ∝ I²
We fit a second-degree polynomial regression on Current vs Conductor Loss and evaluate the coefficient of determination $R^2$.


In [ ]:
sample_fdr = df_meas[df_meas["feeder_id"] == "FDR_01"].sample(1500, random_state=42)
x = sample_fdr["current_avg"].values
y = sample_fdr["feeder_loss_kw"].values

# Fit quadratic model y = a * x^2 + b
poly_coeffs = np.polyfit(x ** 2, y, 1)
y_pred = poly_coeffs[0] * (x ** 2) + poly_coeffs[1]
slope, intercept, r_value, p_value, std_err = stats.linregress(x ** 2, y)

print(f"Quadratic Regression: P_loss = {poly_coeffs[0]:.6f} * I^2 + {poly_coeffs[1]:.4f}")
print(f"Coefficient of Determination R^2 = {r_value**2:.4f}")

plt.figure(figsize=(9, 5))
plt.scatter(x, y, alpha=0.3, color="#3182ce", label="Telemetry Data Points")
sort_idx = np.argsort(x)
plt.plot(x[sort_idx], y_pred[sort_idx], color="#e53e3e", linewidth=2.5, label=f"Physical Fit (R² = {r_value**2:.4f})")
plt.title("Empirical Verification of Joule's First Law: P_loss ∝ I² on Feeder FDR_01", fontsize=12, fontweight="bold")
plt.xlabel("Feeder Current I_avg (Amperes)", fontsize=11)
plt.ylabel("Feeder Technical Loss (kW)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()


## 3. Conductor Resistance vs Percentage Energy Loss
We illustrate how conductor selection and line length dictate technical efficiency.


In [ ]:
plt.figure(figsize=(9, 4.5))
sns.regplot(data=loss_summary, x="total_resistance_ohm", y="Avg_Loss_Pct", color="#805ad5", scatter_kws={"s": 80})
for _, row in loss_summary.iterrows():
    plt.text(row["total_resistance_ohm"] + 0.1, row["Avg_Loss_Pct"], row["feeder_id"], fontsize=9)
plt.title("Average Loss Percentage vs Total Feeder Conductor Resistance R (Ω)", fontsize=12, fontweight="bold")
plt.xlabel("Total Line Resistance (Ohms)")
plt.ylabel("Average Technical Loss %")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()
